In [47]:
import pandas as pd
import numpy as np

# --- Загрузка данных ---
# Файл с предсказаниями: колонки без названий, поэтому задаём имена сами
df_pred = pd.read_csv("x_test_pred.csv", names=["id", "pred"], header=0)  # header=0 если есть заголовок? Уточним
# Если в файле нет заголовка, используйте header=None
# df_pred = pd.read_csv("x_test_pred.csv", names=["id", "pred"], header=None)

# Файл Б с реальными высотами
df_true = pd.read_csv("cup_it_example_src_B_cleaned.csv", usecols=["height_numeric"])
df_true = df_true.reset_index().rename({"index": "id"}, axis=1)

# --- Объединение по id (только те, для которых есть предсказания) ---
merged = pd.merge(df_true, df_pred, on="id", how="inner")

# Убедимся, что высоты числовые
merged["height_numeric"] = pd.to_numeric(merged["height_numeric"], errors="coerce")
merged["pred"] = pd.to_numeric(merged["pred"], errors="coerce")

# Удаляем строки с пропусками, если они есть
merged = merged.dropna(subset=["height_numeric", "pred"])

# Вычисляем разницу (предсказание - реальность)
merged["diff"] = merged["pred"] - merged["height_numeric"]

# --- Создание интервалов по реальной высоте ---
# Задаём границы интервалов. Например, от 0 до 100 с шагом 10.
# При необходимости измените bins под ваши данные (можно вычислить автоматически)

merged.head()

,id,height_numeric,pred,diff
0,5,6.6,8.189452,1.589452
1,8,6.6,8.261693,1.661693
2,10,6.6,6.576164,-0.023836
3,12,6.6,8.899185,2.299185
4,17,9.9,6.674326,-3.225675


In [48]:
df = pd.read_csv('test_pred.csv')
df.head()

,test,pred
0,15.0,19.077243
1,13.2,13.842444
2,30.0,29.682576
3,6.6,15.377070
4,8.0,8.681829


In [49]:
bins = list(range(0, 101, 10))
labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]

# Добавляем колонку с интервалом
df["height_interval"] = pd.cut(df["test"], bins=bins, labels=labels, right=False)
df.head()

,test,pred,height_interval
0,15.0,19.077243,10-20
1,13.2,13.842444,10-20
2,30.0,29.682576,30-40
3,6.6,15.377070,0-10
4,8.0,8.681829,0-10


In [50]:
def msle(y_pred, y_test, c):
    return np.square(np.log(y_test + c) - np.log(y_pred + c)).sum() / len(y_pred)

def rmsle(y_pred, y_test, c):
    return np.sqrt(msle(y_pred, y_test, c))

def get_metrics(y_pred, y_test):
    n = len(y_pred)
    abs_delta = np.abs(y_pred - y_test)

    metrics = {
        'MAE': abs_delta.sum() / n,
        'MAPE': (abs_delta / y_test).sum() / n,
        'WAPE': abs_delta.sum() / y_test.sum(),
        'MSE': (abs_delta ** 2).sum() / n,
        'RMSE': np.sqrt((abs_delta ** 2).sum() / n)
    }

    #for ci in c:
    #    metrics[f'MSLE_{ci}'] = msle(y_pred, y_test, ci)
    #    metrics[f'RMSLE_{ci}'] = rmsle(y_pred, y_test, ci)

    return metrics

In [51]:
rows = {}

for group, subset in df.groupby('height_interval'):
    met = get_metrics(subset['pred'], subset['test'])
    rows[group] = met

In [52]:
df_metrics = pd.DataFrame.from_dict(rows, orient='index')

In [53]:
df_metrics

,MAE,MAPE,WAPE,MSE,RMSE
0-10,2.085676,0.275198,0.262126,9.459648,3.075654
10-20,2.184944,0.147703,0.149422,10.127647,3.182396
20-30,5.810206,0.246927,0.242441,55.348167,7.439635
30-40,4.362062,0.130823,0.133030,46.912608,6.849278
40-50,6.176908,0.138426,0.138682,85.338457,9.237882
50-60,8.406998,0.157047,0.157316,138.806329,11.781610
60-70,14.539822,0.230052,0.231076,403.143565,20.078435
70-80,19.455294,0.260243,0.259717,677.269864,26.024409
80-90,22.941899,0.272852,0.272859,882.116757,29.700450
90-100,19.711609,0.214263,0.213851,585.872691,24.204807


In [54]:
df_metrics.to_csv('metrics.csv')

In [9]:
df['delta'] = df['test'] - df['pred']
df['abs_delta'] = np.abs(df['delta'])

df.head()

,test,pred,delta,abs_delta
0,15.0,20.464590,-5.464590,5.464590
1,13.2,13.529882,-0.329882,0.329882
2,30.0,29.770056,0.229944,0.229944
3,6.6,15.518620,-8.918620,8.918620
4,8.0,7.839814,0.160186,0.160186
